1/x in encoder1 4670.44
1/x in encoder2 6514.89

Conclusion
6514.89 -> introduce a lot of error

In [1]:
import subprocess
from datasets import load_dataset
import re
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

dataset = load_dataset("dair-ai/emotion", split="test")

/Users/tonyma/code/FHE-BERT-Tiny-Emotion/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gokuls/BERT-tiny-emotion-intent")

texts = dataset['text']
token_length = [len(tokenizer.encode(entry['text'], truncation=False)) for entry in dataset]
labels = dataset['label']

df = pd.DataFrame({
    'text': texts,
    'token_length': token_length,
    'label': labels
})

# sort label
df_label = df.sort_values(by=['label', 'token_length'], ascending=[True, True]).reset_index(drop=True)

In [3]:
new_df2 = df_label.groupby('label').head(10).reset_index(drop=True)
print(len(new_df2))
print(new_df2[:21])

60
                            text  token_length  label
0          i feel so embarrassed             6      0
1             i do feel stressed             6      0
2                i feels so lame             6      0
3      i feel embarrassed enough             6      0
4          i stop feeling guilty             6      0
5         i feel depressed again             6      0
6              i feel soo lonely             6      0
7     im feeling depressed again             6      0
8           i just feel troubled             6      0
9              i feel less alone             6      0
10          im feeling energetic             5      1
11          i feel more creative             6      1
12        i feel really inspired             6      1
13         i feel more energetic             6      1
14             i feel any better             6      1
15       i would feel productive             6      1
16   im feeling hopeful relieved             6      1
17           i feel gorge

In [4]:
correct = []
error = []
output = []
execution_times = []

func_error = []
count = 0

def run_fhe_bert_tiny(sample):
    global correct, error, func_error, output, execution_times, count
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    cmd = [f"./FHE-BERT-Tiny", sentence]
    #cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct.append(count)
            print("sadness")
        else:
            error.append(count)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct.append(count)
            print("joy")
        else:
            error.append(count)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct.append(count)
            print("love")
        else:
            error.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct.append(count)
            print("anger")
        else:
            error.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct.append(count)
            print("fear")
        else:
            error.append(count)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct.append(count)
            print("surprise")
        else:
            error.append(count)
    else: # error
        func_error.append(count)
        
    print(seconds, "seconds")

In [5]:
from tqdm import tqdm

for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    run_fhe_bert_tiny(row)
    count+=1

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [02:29<2:27:14, 149.73s/it]

[-1.19874e+34, 3.70629e+32, 2.96691e+33, 8.35415e+33, 2.33337e+33, -5.86724e+33]
136.0 seconds
sentence:  i do feel stressed


  3%|▎         | 2/60 [04:56<2:23:11, 148.14s/it]

...

[1.64284e+33, 3.78551e+33, 1.18955e+34, 1.02671e+34, -6.20167e+33, 3.83962e+33]
182.0 seconds
sentence:  i am feeling amazing and seeing the difference


 98%|█████████▊| 59/60 [2:44:31<03:09, 189.24s/it]

[-1.76071e+33, 8.4328e+33, 1.0055e+34, 3.52876e+33, 5.82855e+33, 3.53168e+33]
183.0 seconds
sentence:  i always feel very shocked by that me threatening


100%|██████████| 60/60 [2:48:00<00:00, 168.02s/it]

[-3.33103e+33, -3.16938e+32, 2.11795e+33, -5.04819e+33, 7.6768e+33, 4.99137e+33]
198.0 seconds


BELOW NEED CHECK & MODIFY

In [6]:
len(correct)

19

In [7]:
len(error)

41

In [8]:
error

[0,
 2,
 3,
 4,
 7,
 12,
 16,
 17,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 31,
 32,
 33,
 35,
 36,
 38,
 40,
 41,
 42,
 43,
 44,
 47,
 48,
 49,
 51,
 52,
 53,
 54,
 56,
 57,
 58,
 59]

In [9]:
print(len(output))
print(len(execution_times))

60
60


In [10]:
import numpy as np
np.savetxt("output_labels_60", output, delimiter=",")
np.savetxt("execution_time_labels_60", output, delimiter=",")

Test again

In [11]:
import math

new_resamples = []
for i in range(len(output)):
    if (int(math.floor(math.log10(abs(output[i][0])))) > 0) or (int(math.floor(math.log10(abs(output[i][1])))) > 0) or (int(math.floor(math.log10(abs(output[i][2])))) > 0) or (int(math.floor(math.log10(abs(output[i][3])))) > 0) or (int(math.floor(math.log10(abs(output[i][4])))) > 0) or (int(math.floor(math.log10(abs(output[i][5])))) > 0):
        print(f"{i}: {output[i]}")
        new_resamples.append(i)
        
print(len(new_resamples))

0: [-1.19874e+34, 3.70629e+32, 2.96691e+33, 8.35415e+33, 2.33337e+33, -5.86724e+33]
2: [-5.75694e+33, -1.37762e+33, 4.90461e+33, 1.51909e+33, 7.81159e+33, 4.08901e+33]
3: [2.20566e+33, 2.17105e+34, -1.71892e+34, -7.13193e+33, -1.92507e+33, 5.14462e+33]
4: [-3.63047e+33, -3.93144e+33, -9.28306e+33, -5.37346e+33, -8.22563e+32, 1.5982e+32]
7: [-1.31936e+19, 1.20078e+19, 1.17972e+19, -4.11204e+19, -9.54204e+18, 5.12544e+18]
8: [1.94244e+34, 8.32377e+33, -4.99101e+32, -6.82477e+33, 2.83086e+33, 2.7326e+32]
9: [1.15683e+34, -9.01331e+33, 5.89831e+33, -1.01301e+34, 1.86831e+33, -5.00992e+33]
10: [-8.84361e+19, 8.48007e+19, 2.48987e+18, 2.94897e+19, -4.34959e+17, -2.31193e+19]
12: [7.51284e+32, -1.83333e+33, 3.72409e+33, 7.56881e+32, 1.36544e+34, 9.12972e+31]
16: [-1.25411e+34, -1.00451e+34, -7.59937e+33, -4.05653e+33, 5.51399e+33, -5.91543e+33]
20: [1.21605e+34, -5.25196e+33, -3.58989e+33, -1.14932e+34, -1.82026e+33, -8.55523e+33]
21: [1.71051e+34, -7.93955e+33, -2.68513e+33, 2.63256e+33, -1.

In [12]:
new_resamples

[0,
 2,
 3,
 4,
 7,
 8,
 9,
 10,
 12,
 16,
 20,
 21,
 26,
 27,
 29,
 32,
 33,
 34,
 35,
 36,
 38,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 53,
 54,
 57,
 58,
 59]

In [13]:
for i in new_resamples:
    if not i in error:
        print("not exist: ", i)

not exist:  8
not exist:  9
not exist:  10
not exist:  34
not exist:  45
not exist:  46
not exist:  50


In [14]:
correct2 = []
error2 = []
output2 = []
execution_times2 = []

func_error2 = []
count2 = 0

def run_fhe_bert_tiny2(sample):
    global correct2, error2, func_error2, output2, execution_times2, count2
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    #cmd = [f"./FHE-BERT-Tiny", sentence]
    cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output2.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times2.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct2.append(count2)
            print("sadness")
        else:
            error2.append(count2)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct2.append(count2)
            print("joy")
        else:
            error2.append(count2)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct2.append(count2)
            print("love")
        else:
            error2.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct2.append(count2)
            print("anger")
        else:
            error2.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct2.append(count2)
            print("fear")
        else:
            error2.append(count2)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct2.append(count2)
            print("surprise")
        else:
            error2.append(count2)
    else: # error
        error2.append(count2)
        
    print(seconds, "seconds")

In [15]:
from tqdm import tqdm

eliminated = 0
for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    if count2 == new_resamples[eliminated]:
        #print(count)
        eliminated+=1
        run_fhe_bert_tiny2(row)
    count2+=1
    
    if eliminated == len(new_resamples):
        break

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [02:24<2:22:07, 144.53s/it]

[-2.727478095796478e+33, -6.998759259923465e+32, 5.87548220886416e+33, -1.4253735819599046e+34, 5.980753830017431e+33, 8.53266521533564e+33]
133.0 seconds
sentence:  i feels so lame


  5%|▌         | 3/60 [04:54<1:28:13, 92.87s/it] 

...

[-9.478176125225093e+33, 3.9891412735495786e+33, 4.770102903475804e+33, 1.6308385542258242e+34, -2.2111165584255644e+33, -1.7355391571626464e+34]
187.0 seconds
sentence:  i am feeling amazing and seeing the difference


 98%|█████████▊| 59/60 [1:39:04<02:00, 120.71s/it]

[1.0648670537290627e+33, 8.691649638719455e+33, -2.999622436624415e+33, -1.9321311191890736e+34, 5.249891463654877e+33, 6.389092209826146e+33]
185.0 seconds
sentence:  i always feel very shocked by that me threatening


 98%|█████████▊| 59/60 [1:42:27<01:44, 104.20s/it]

[3.5786947855558025e+33, 5.0756780959005323e+33, 1.6889676209756267e+34, 1.203602725562282e+34, -7.86562008983627e+32, 1.6456838163567451e+34]
192.0 seconds


In [16]:
len(correct2)

8

In [17]:
# total accuracy
total_accuracy = correct + correct2
total_accuracy.sort()
print(len(total_accuracy))

27


In [18]:
len(total_accuracy)/60

0.45

In [19]:
# Each label accuracy
label0_accuracy = []
label1_accuracy = []
label2_accuracy = []
label3_accuracy = []
label4_accuracy = []
label5_accuracy = []

for i in total_accuracy:
    if i < 10:
        label0_accuracy.append(i)
    elif i < 20:
        label1_accuracy.append(i)
    elif i < 30:
        label2_accuracy.append(i)
    elif i < 40:
        label3_accuracy.append(i)
    elif i < 50:
        label4_accuracy.append(i)
    elif i < 60:
        label5_accuracy.append(i)

In [20]:
print(len(label0_accuracy))
print(len(label1_accuracy))
print(len(label2_accuracy))
print(len(label3_accuracy))
print(len(label4_accuracy))
print(len(label5_accuracy))

8
8
0
5
3
3


In [21]:
print(df_label[df_label['label'] == 0].shape[0])
print(df_label[df_label['label'] == 1].shape[0])
print(df_label[df_label['label'] == 2].shape[0])
print(df_label[df_label['label'] == 3].shape[0])
print(df_label[df_label['label'] == 4].shape[0])
print(df_label[df_label['label'] == 5].shape[0])


581
695
159
275
224
66


In [22]:
new_resamples

[0,
 2,
 3,
 4,
 7,
 8,
 9,
 10,
 12,
 16,
 20,
 21,
 26,
 27,
 29,
 32,
 33,
 34,
 35,
 36,
 38,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 53,
 54,
 57,
 58,
 59]

In [23]:
# check final wrong answer
final_wrong = []

for i in range(60):
    if not i in total_accuracy and i in new_resamples:
        print(i)
        print(f"first: {output[i]}")
        print(f"retest: {output2[new_resamples.index(i)]}")
        final_wrong.append(i)

0
first: [-1.19874e+34, 3.70629e+32, 2.96691e+33, 8.35415e+33, 2.33337e+33, -5.86724e+33]
retest: [-2.727478095796478e+33, -6.998759259923465e+32, 5.87548220886416e+33, -1.4253735819599046e+34, 5.980753830017431e+33, 8.53266521533564e+33]
2
first: [-5.75694e+33, -1.37762e+33, 4.90461e+33, 1.51909e+33, 7.81159e+33, 4.08901e+33]
retest: [-9.40246836915526e+33, 7.484271901682712e+33, -2.7609753321921465e+32, -9.939711971648592e+32, 1.1745495285300646e+34, -3.311332417095802e+33]
3
first: [2.20566e+33, 2.17105e+34, -1.71892e+34, -7.13193e+33, -1.92507e+33, 5.14462e+33]
retest: [1.651826681496213e+33, 5.263146172657339e+33, 1.5324016384376162e+33, -5.367404027268024e+33, 1.341859010588342e+34, -6.308982293483076e+33]
16
first: [-1.25411e+34, -1.00451e+34, -7.59937e+33, -4.05653e+33, 5.51399e+33, -5.91543e+33]
retest: [1.487422769131787e+34, -1.1830724438815516e+34, 7.765013782161336e+33, 3.075292494938566e+33, -7.069695209530323e+33, 1.2599823197903316e+33]
20
first: [1.21605e+34, -5.25196e

In [24]:
print(len(final_wrong))

24


In [ ]:
import numpy as np
np.savetxt("output2_labels_60", output, delimiter=",")
np.savetxt("execution_time2_labels_60", output, delimiter=",")